In [1]:
import glob
import vtk
import numpy as np
from vtk.util import numpy_support as VN
import matplotlib.pyplot as plt

In [2]:
# simulation parameters
DP = 1                       # pressure drop [Pa]
nu = 1e-06                   # kinematic viscosity [m²/s]
rho = 1000                   # density [kg/m³]
mu = nu * rho                # dynamic viscosity [kg/(m*s)]
dx = 1256*1e-6                # model length x [m]
dy = 1256*1e-6                # model width y [m]
dz = 1e-6                    # model thickness z [m]
A = dy * dz
V = dx * A



# load VTK data
# the VTK file written by foamToVTK, e.g. ../VTK/RandomDisks_557.vtk
vtkFile = sorted(glob.glob('../VTK/*.vtk'))[-1]
print(f'Reading {vtkFile}')
reader = vtk.vtkUnstructuredGridReader()
reader.SetFileName(vtkFile)
reader.ReadAllScalarsOn()
reader.ReadAllVectorsOn()
reader.Update()
data = reader.GetOutput()


# extract velocity arrays
Vcells = VN.vtk_to_numpy(data.GetCellData().GetArray('V'))
U = VN.vtk_to_numpy(data.GetCellData().GetArray('U'))
Umag = np.sqrt(U[:,0]**2+U[:,1]**2+U[:,2]**2)
Ux = U[:,0]
Uy = U[:,1]
Uz = U[:,2]

# calculate volume fluxes
qx = []
qy = []
for c in range(len(Vcells)):
    q1 = Vcells[c] * Ux[c]
    q2 = Vcells[c] * Uy[c]
    qx.append(q1)
    qy.append(q2)

# calculate Darcy velocity
U_Darcy_x = np.sum(qx)/V

kxx = mu * dx * U_Darcy_x/DP
porosity = sum(Vcells)/V

# print permeability and porosity

print('Bulk permeability: %5.2e m2, Porosity: %3.2f' % (kxx, porosity))

Reading ../VTK/RandomDisks_525.vtk
Bulk permeability: 1.95e-11 m2, Porosity: 0.59
